# Excusify: fine-tuning GPT-2 for situation-aware excuses, apologies and emergency messages

This notebook trains the three models used by the Excusify app and uploads them to the Hugging Face Hub.

**Run it on a GPU:** *Runtime → Change runtime type → T4 GPU*. All three models train in about 10 minutes.

What changed from v1 (and why the old models produced off-topic text):

1. **The data now says what each excuse is for.** Every row has a `Situation` (e.g. *forgot to submit the report*), so the model learns to write an excuse *for that situation* instead of a random one.
2. **The model learns when to stop.** v1 set `pad_token = eos_token` and used `DataCollatorForLanguageModeling`, which masks every pad token, so every end-of-text label was ignored and the model never learned to stop. Here the loss is computed on the completion **plus** its end-of-text token.
3. **Loss only on the completion.** Prompt tokens are masked (`-100`), so training effort goes into writing good excuses, not into memorising the prompt format.
4. **Real evaluation.** 10% of each dataset is held out; we report eval loss and perplexity before and after fine-tuning and keep the best epoch.
5. **The prompt format is shared with `app.py`.** The functions in step 2 are copied verbatim into the app, so training and inference see exactly the same text.

In [ ]:
!pip install -q "transformers>=4.41" accelerate huggingface_hub scikit-learn

## 1. Configuration

In [ ]:
HF_USERNAME = "Sohamb2005"     # your Hugging Face username
PUSH_TO_HUB = True             # set to False to train and test without uploading
BASE_MODEL = "gpt2"
DATA_URL = "https://raw.githubusercontent.com/soham200521/Excusify/main/data"
MAX_LENGTH = 96                # the longest example is checked against this below
EPOCHS = 5
LEARNING_RATE = 5e-5
BATCH_SIZE = 8
SEED = 42

MODELS = {
    "excuse": f"{HF_USERNAME}/gpt2-finetuned-excuses",
    "apology": f"{HF_USERNAME}/gpt2-finetuned-apologies",
    "emergency": f"{HF_USERNAME}/gpt2-finetuned-emergency",
}

## 2. Prompt format (shared with `app.py`; keep the two copies identical)

In [ ]:
import re

def normalize(text):
    """Lower-case, straight quotes, no separator characters, no trailing punctuation."""
    text = (text or "").replace("\u2019", "'").replace("\u2018", "'").replace("\u201c", '"').replace("\u201d", '"')
    text = re.sub(r"[|:\n]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip().rstrip(".!?").strip()
    return text.lower()


def excuse_prompt(scenario, urgency, believability, situation):
    return f"{normalize(scenario)} | {normalize(urgency)} | {normalize(believability)} | {normalize(situation)} :"


def apology_prompt(apology_type, situation):
    return f"{normalize(apology_type)} | {normalize(situation)} :"


def emergency_prompt(scenario, details=""):
    details = normalize(details)
    return f"{normalize(scenario)} | {details} :" if details else f"{normalize(scenario)} :"

## 3. Load the datasets

In [ ]:
import os
import pandas as pd

def load_csv(name):
    local = os.path.join("data", name)
    path = local if os.path.exists(local) else f"{DATA_URL}/{name}"
    df = pd.read_csv(path, keep_default_na=False)
    print(f"{name}: {len(df)} rows  ({path})")
    return df


def to_pairs(df, prompt_fn, text_col, group_col):
    pairs = pd.DataFrame({
        "prompt": df.apply(prompt_fn, axis=1),
        "completion": df[text_col].str.strip(),
        "group": df[group_col],
    })
    before = len(pairs)
    pairs = pairs.drop_duplicates(subset=["completion"]).reset_index(drop=True)
    if before != len(pairs):
        print(f"  dropped {before - len(pairs)} duplicate rows")
    return pairs


excuses = load_csv("excuses.csv")
apologies = load_csv("apologies.csv")
emergencies = load_csv("emergency_messages.csv")

datasets = {
    "excuse": to_pairs(excuses, lambda r: excuse_prompt(r.Scenario, r.Urgency, r.Believability, r.Situation), "Excuse", "Scenario"),
    "apology": to_pairs(apologies, lambda r: apology_prompt(r.Type, r.Situation), "Apology", "Type"),
    "emergency": to_pairs(emergencies, lambda r: emergency_prompt(r.Scenario, r.Details), "Message", "Scenario"),
}
for name, pairs in datasets.items():
    print(f"\n{name}:  {pairs.prompt[0]} {pairs.completion[0]}")

## 4. Tokenise with prompt masking and an end-of-text label

In [ ]:
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token


class PromptCompletionDataset(Dataset):
    """Loss is computed on the completion and its EOS token only; prompt and padding are -100."""

    def __init__(self, frame):
        self.items = [self.encode(p, c) for p, c in zip(frame.prompt, frame.completion)]

    @staticmethod
    def encode(prompt, completion):
        prompt_ids = tokenizer(prompt)["input_ids"]
        completion_ids = tokenizer(" " + completion)["input_ids"] + [tokenizer.eos_token_id]
        ids = prompt_ids + completion_ids
        if len(ids) > MAX_LENGTH:
            raise ValueError(f"Example is {len(ids)} tokens, above MAX_LENGTH={MAX_LENGTH}: {prompt} {completion}")
        pad = MAX_LENGTH - len(ids)
        return {
            "input_ids": torch.tensor(ids + [tokenizer.pad_token_id] * pad),
            "attention_mask": torch.tensor([1] * len(ids) + [0] * pad),
            "labels": torch.tensor([-100] * len(prompt_ids) + completion_ids + [-100] * pad),
        }

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


splits = {}
for name, pairs in datasets.items():
    train_df, eval_df = train_test_split(pairs, test_size=0.1, random_state=SEED, stratify=pairs["group"])
    train_ds, eval_ds = PromptCompletionDataset(train_df), PromptCompletionDataset(eval_df)
    longest = max(int(x["attention_mask"].sum()) for x in train_ds.items + eval_ds.items)
    splits[name] = (train_ds, eval_ds)
    print(f"{name:9s}  train={len(train_ds):4d}  eval={len(eval_ds):3d}  longest example={longest} tokens")

# sanity check: the label sequence of the first example ends with the EOS token
first = splits["excuse"][0][0]
assert tokenizer.eos_token_id in first["labels"].tolist(), "EOS must be a training target"
print("EOS is a training target:", True)

## 5. Train (best epoch kept by eval loss)

In [ ]:
import inspect
import math
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, default_data_collator, set_seed


def make_args(output_dir):
    kwargs = dict(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=20,
        report_to="none",
        seed=SEED,
        fp16=torch.cuda.is_available(),
    )
    # argument names differ between transformers 4.x and 5.x
    params = inspect.signature(TrainingArguments.__init__).parameters
    kwargs["eval_strategy" if "eval_strategy" in params else "evaluation_strategy"] = "epoch"
    if "warmup_ratio" in params:
        kwargs["warmup_ratio"] = 0.1      # transformers 4.x
    else:
        kwargs["warmup_steps"] = 0.1      # transformers 5.x: a float below 1 is a ratio
    return TrainingArguments(**kwargs)


def train(name):
    set_seed(SEED)
    train_ds, eval_ds = splits[name]
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
    model.config.pad_token_id = tokenizer.eos_token_id
    trainer = Trainer(model=model, args=make_args(f"./runs/{name}"), train_dataset=train_ds,
                      eval_dataset=eval_ds, data_collator=default_data_collator)
    base_loss = trainer.evaluate()["eval_loss"]
    trainer.train()
    loss = trainer.evaluate()["eval_loss"]
    metrics = {
        "train_examples": len(train_ds), "eval_examples": len(eval_ds),
        "base_eval_loss": round(base_loss, 3), "eval_loss": round(loss, 3),
        "base_perplexity": round(math.exp(base_loss), 1), "perplexity": round(math.exp(loss), 1),
    }
    print(name, metrics)
    return trainer, metrics


results = {name: train(name) for name in ["excuse", "apology", "emergency"]}

## 6. Try the models (including situations that are not in the training data)

In [ ]:
GEN = dict(do_sample=True, top_p=0.92, top_k=50, temperature=0.8, max_new_tokens=60,
           no_repeat_ngram_size=3, repetition_penalty=1.1)


def generate(model, prompt, n=3):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, num_return_sequences=n, pad_token_id=tokenizer.eos_token_id,
                         eos_token_id=tokenizer.eos_token_id, **GEN)
    start = inputs["input_ids"].shape[1]
    return [tokenizer.decode(o[start:], skip_special_tokens=True).strip() for o in out]


SAMPLES = {
    "excuse": [
        excuse_prompt("work", "medium", "high", "Forgot to submit the report"),
        excuse_prompt("school", "high", "low", "missed the exam"),
        excuse_prompt("social", "low", "high", "late for my friend's birthday dinner"),   # not in the data
        excuse_prompt("family", "medium", "medium", "forgot to water the plants"),        # not in the data
    ],
    "apology": [
        apology_prompt("professional", "being late to the meeting"),
        apology_prompt("emotional", "forgetting our anniversary"),
        apology_prompt("informal", "eating your leftover pizza"),                         # not in the data
    ],
    "emergency": [
        emergency_prompt("car trouble", "flat tyre on the highway"),
        emergency_prompt("medical issue"),
        emergency_prompt("stuck somewhere", "stuck at the airport after a cancelled flight"),  # not in the data
    ],
}

for name, prompts in SAMPLES.items():
    model = results[name][0].model.eval()
    print(f"\n===== {name} =====")
    for prompt in prompts:
        print(prompt)
        for text in generate(model, prompt):
            print("   ->", text)

## 7. Save, write model cards and upload to the Hub

Skip this section if `PUSH_TO_HUB = False`. You will be asked for a Hugging Face **write** token (Settings → Access Tokens).

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
from transformers import GenerationConfig

DESCRIPTIONS = {
    "excuse": ("excuses", "an excuse for a situation, conditioned on scenario, urgency and believability",
               'excuse_prompt("work", "medium", "high", "forgot to submit the report")',
               "work | medium | high | forgot to submit the report :", "excuses.csv",
               "Scenario (work/school/family/social), Situation, Urgency (low/medium/high), Believability (low/medium/high), Excuse"),
    "apology": ("apologies", "an apology for a situation in an emotional, professional or informal tone",
                'apology_prompt("professional", "being late to the meeting")',
                "professional | being late to the meeting :", "apologies.csv",
                "Type (emotional/professional/informal), Situation, Apology"),
    "emergency": ("emergency messages", "a short, first-person urgent message for a scenario and optional details",
                  'emergency_prompt("car trouble", "flat tyre on the highway")',
                  "car trouble | flat tyre on the highway :", "emergency_messages.csv",
                  "Scenario (8 types), Details (optional), Message"),
}


def model_card(name, metrics):
    what, task, call, prompt, csv_name, columns = DESCRIPTIONS[name]
    return f"""---
license: mit
language: en
base_model: gpt2
pipeline_tag: text-generation
tags: [gpt2, text-generation, fine-tuned, excusify]
---

# GPT-2 fine-tuned for {what} (Excusify)

Part of [Excusify](https://github.com/soham200521/Excusify). Given a short prompt, the model writes {task}.

## Prompt format

```
{prompt}
```

The model writes the completion after the colon and stops with an end-of-text token.

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

repo = "{MODELS[name]}"
tokenizer = AutoTokenizer.from_pretrained(repo)
model = AutoModelForCausalLM.from_pretrained(repo)

inputs = tokenizer("{prompt}", return_tensors="pt")
out = model.generate(**inputs, do_sample=True, top_p=0.92, temperature=0.8, max_new_tokens=60,
                     pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
```

## Training

| | |
|---|---|
| Base model | `gpt2` (124M parameters) |
| Data | [`data/{csv_name}`](https://github.com/soham200521/Excusify/tree/main/data): {columns} |
| Split | {metrics["train_examples"]} train / {metrics["eval_examples"]} held-out examples |
| Objective | causal LM loss on the completion and its end-of-text token (prompt masked) |
| Epochs / LR / batch | {EPOCHS} / {LEARNING_RATE} / {BATCH_SIZE}, cosine schedule, best epoch by eval loss |

## Evaluation (held-out set)

| | Eval loss | Perplexity |
|---|---|---|
| GPT-2 before fine-tuning | {metrics["base_eval_loss"]} | {metrics["base_perplexity"]} |
| This model | {metrics["eval_loss"]} | {metrics["perplexity"]} |

## Limitations

Made for entertainment and practice messages. The training set is small and hand-written, so the model can repeat
phrasings from it, and situations far from the training data may get generic answers.
"""


if PUSH_TO_HUB:
    from huggingface_hub import HfApi
    api = HfApi()

for name, (trainer, metrics) in results.items():
    out_dir = "./" + MODELS[name].split("/")[-1]
    trainer.model.generation_config = GenerationConfig(**GEN, pad_token_id=tokenizer.eos_token_id,
                                                       eos_token_id=tokenizer.eos_token_id)
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)
    with open(os.path.join(out_dir, "README.md"), "w") as f:
        f.write(model_card(name, metrics))
    print("saved", out_dir)
    if PUSH_TO_HUB:
        api.create_repo(MODELS[name], exist_ok=True)
        api.upload_folder(folder_path=out_dir, repo_id=MODELS[name],
                          commit_message="v2: situation-aware data, EOS fix, held-out evaluation")
        print("uploaded", MODELS[name])

## 8. Metrics table for the README

In [ ]:
print("| Model | Train / eval examples | Eval loss (GPT-2 → fine-tuned) | Perplexity (GPT-2 → fine-tuned) |")
print("|---|---|---|---|")
for name, (_, m) in results.items():
    print(f"| `{MODELS[name]}` | {m['train_examples']} / {m['eval_examples']} | "
          f"{m['base_eval_loss']} → {m['eval_loss']} | {m['base_perplexity']} → {m['perplexity']} |")